# JarvisLM-350M: A100 10B-token pretraining

目标：在单张 NVIDIA A100 40GB 上完整运行 JarvisLM-350M 的 FineWeb-Edu 10B-token 预训练。模型结构、tokenizer、优化器、学习率日程、EMA、有效 global batch、checkpoint、W&B 与 H100 正式路线一致；仅将 micro-batch 调整为 A100 已验证的 2，并把梯度累积设为 256，以保持每个 update 的 524,288 tokens 不变。

不要在本 notebook 写入 token。它使用隐藏输入，仅保存模型 checkpoint 和数据 shard。

## 运行前条件

- A100 GPU，建议 40GB 或更多显存；本 notebook 已在 A100 40GB 上验证 350M smoke。
- `/content` 或挂载卷至少留出 45GB：约 20GB 数据、约 14GB checkpoint、缓存余量。
- 10B tokens 是长任务。请确认 IDE/Colab 的磁盘与 session 可持久；中断后不要删除 checkpoint，重新运行正式训练 cell 会自动续训。
- 这不是 H100 结果：最终简历应写 A100，吞吐/MFU/loss/PPL 必须以这次真实训练记录为准。

In [ ]:
import os
from pathlib import Path
import torch

assert torch.cuda.is_available(), '请先连接 A100 GPU。'
gpu_name = torch.cuda.get_device_name(0)
assert 'A100' in gpu_name.upper(), f'该 notebook 目标是 A100，当前为 {gpu_name}'
free_gb = os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 2**30
assert free_gb >= 45, f'可用磁盘仅 {free_gb:.1f} GB；请先挂载更大磁盘。'
print({'gpu': gpu_name, 'gpu_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'disk_free_gb': round(free_gb, 1), 'bf16': torch.cuda.is_bf16_supported()})

## 安全加载 token（IDE 模式）

输入隐藏且只存在于本次 kernel。请使用新生成的 token；不要把值写入 notebook 或 terminal 历史。

In [ ]:
from getpass import getpass

def require_secret(name: str) -> None:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name} (hidden input): ')
    if not os.environ[name]:
        raise RuntimeError(f'{name} is required')

require_secret('HF_TOKEN')
require_secret('WANDB_API_KEY')
print({'hf_token_loaded': bool(os.environ.get('HF_TOKEN')), 'wandb_api_key_loaded': bool(os.environ.get('WANDB_API_KEY'))})

## 拉取项目与固定实验配置

此 cell 可重复运行，会拉取训练分支的最新提交。

In [ ]:
%cd /content
!if [ -d small-llm-from-scratch/.git ]; then git -C small-llm-from-scratch fetch origin codex/jarvislm-350m-training && git -C small-llm-from-scratch checkout codex/jarvislm-350m-training && git -C small-llm-from-scratch pull --ff-only; else git clone --branch codex/jarvislm-350m-training https://github.com/JarvisZhang24/small-llm-from-scratch.git; fi
%cd /content/small-llm-from-scratch
!python -m pip install -q -e '.[dev]'
RUN_ROOT = Path('/content/jarvislm-a100-10b-v1')
TRAIN_DIR = RUN_ROOT / 'data/train'
VAL_DIR = RUN_ROOT / 'data/val'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
TRAIN_TOKENS = 10_000_000_000
VAL_TOKENS = 20_000_000
MICRO_BATCH = 2
GRAD_ACCUM = 256
MAX_STEPS = 19_074
TOKENS_PER_UPDATE = 1024 * MICRO_BATCH * GRAD_ACCUM
assert TOKENS_PER_UPDATE == 524_288
print({'run_root': str(RUN_ROOT), 'train_tokens': TRAIN_TOKENS, 'tokens_per_update': TOKENS_PER_UPDATE, 'max_steps': MAX_STEPS, 'scheduled_tokens': TOKENS_PER_UPDATE * MAX_STEPS})

## 一次性准备 FineWeb-Edu

这一步会流式写入 10B 训练 token 和 20M 验证 token。首次运行耗时较长；若已有 shard，代码会拒绝覆盖，这是预期保护。成功后不要再运行此 cell。

In [ ]:
!PYTHONPATH=src python -m jarvislm.training.train --prepare-data --prepare-only --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --prepare-train-tokens {TRAIN_TOKENS} --prepare-val-tokens {VAL_TOKENS}

## 可选：完整有效 batch 的 compile 预检

正式训练前可运行一次。它使用相同的 2×256 累积，但 checkpoint 写在独立目录，不会污染正式实验；首次 compile 很慢是正常的。

In [ ]:
# 取消下一行开头的 # 后运行预检。
# !PYTHONPATH=src python -m jarvislm.training.train --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {RUN_ROOT / 'compile-probe'} --max-steps 1 --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --compile --no-wandb --no-resume

## 正式 A100 10B-token 训练

运行此 cell 开始正式实验。默认每 10 steps 记录 W&B、每 500 steps 验证、每 1000 steps 保存 checkpoint；正常结束或中断后都会写 `last.pt`。重连后重复运行同一个 cell，会从最新 checkpoint 继续。

In [ ]:
!PYTHONPATH=src python -m jarvislm.training.train --run-name jarvislm-350m-a100-10b --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {CHECKPOINT_DIR} --max-steps {MAX_STEPS} --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --compile --wandb --resume

## 读取可恢复状态并记录真实结果

只在训练暂停或完成后运行。将实际 step、tokens、W&B run URL、validation loss、tokens/s、运行时长记录到实验日志；不要根据 smoke 结果预填任何指标。

In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'last.pt'
assert checkpoint_path.is_file(), f'checkpoint missing: {checkpoint_path}'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
completed_steps = int(checkpoint['step'])
print({'completed_steps': completed_steps, 'tokens_seen': completed_steps * TOKENS_PER_UPDATE, 'has_muon': 'muon' in checkpoint, 'has_ema': 'ema' in checkpoint, 'checkpoint': str(checkpoint_path)})

## 训练中断时

保留 `RUN_ROOT`。重新连接 A100 后，依次运行 GPU/token/安装配置 cell，然后直接运行“正式 A100 10B-token 训练”cell；不要再次运行数据准备 cell。若必须做新实验，修改 `RUN_ROOT` 为新目录。